# 01. Base case

**Tier:** Deep. **Why you'd use this:** You walk a countdown or a factorial-style problem where each call needs a smaller input and a clear stop. Without a base case, the calls never end.

**After this lesson you can:** say which lines run on the way down and which run on the way back up; write a base case that makes the input smaller.
About 25 minutes. **Uses:** [Functions](../05-functions/01-definitions-lesson.ipynb), [Loops](../03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What does a function return if it hits the end with no `return`?
2. Can a function call itself? What must eventually happen?

<details><summary>Answers</summary>

It returns `None`. Yes — and some call must stop (the base case) so the stack can unwind.
</details>


## The idea

A function may call itself. The call that stops is the base case. The call that hands a smaller input to the same function is the recursive case. Each call gets its own local names. The caller waits until the callee returns, then finishes its own line.

That waiting is a stack of frames. A frame is the record of one live call: its local names, the line it is on, and who it must return to. The newest frame is on top. A return throws that frame away and resumes the one under it.

<details><summary>Explain more</summary>

The base case is an ordinary `if` that returns without calling the function again. The recursive case must move toward that `if`. For a positive integer, dividing by 10 is a move toward a single digit. If the input does not shrink, the stack grows until Python refuses another frame.

Work before the recursive call runs on the way down, as each frame is created. Work after it runs on the way back up, as each frame returns. Those are not two loops. They are the start and the end of the same frame.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

A self-call is still a normal call. There is no `return recurse(...)` optimization that reuses the current frame. The stack you would draw on a whiteboard is the stack Python keeps.
</details>


## See it work

A meter id is a positive integer. You want how many digits it has, without turning it into text. One digit is done. Anything larger is one digit plus the digits of `n // 10`.


In [1]:
def digit_count(n: int) -> int:
    # 1. A single digit needs no further call.
    if n < 10:
        return 1
    # 2. Count this digit, then the digits that remain.
    return 1 + digit_count(n // 10)


print(digit_count(725))
print(digit_count(8))
print(digit_count(10))
print(digit_count(0))
print(digit_count(-15))

3
1
2
1
1


`725` has three digits, `8` has one, and `10` has two. `0` takes the base case and counts as one digit. `-15` also takes the base case, because every negative number is less than 10, so the function never divides. The base case is the right shape only for the inputs you tested it on. Zero and negatives are different problems wearing the same `if`.


## Check yourself

Answer these from the example above before you run the cell.


1. In `announce`, why does `up 72` print before `up 725`?
2. Why does `digit_count(-15)` return 1?
3. Which frame is still alive while `announce(7)` runs?

<details><summary>Answers</summary>

1. `announce(72)` is the callee. It has to return before the line after the call in `announce(725)` can run. The `up` print is that later line, so the inner frame prints first.
2. The test `n < 10` is true for every negative. The function returns 1 and never reaches `n // 10`. Floor division of a negative would be a different bug. This one never gets that far.
3. The frames for 72 and 725 are both still on the stack. Each is paused on the line that adds 1. The base-case frame is the only one running.
</details>


In [2]:
def announce(n):
    if n < 10:
        print("base", n)
        return 1
    print("down", n)
    result = 1 + announce(n // 10)
    print("up", n, result)
    return result


print("total", announce(725))

down 725
down 72
base 7
up 72 2
up 725 3
total 3


<details><summary>Check your answers</summary>

- `down 725` prints before any call on `72`. The frame for 725 is alive and waiting on the next line.
- `down 72` then `base 7`. The base case prints and returns 1. It does not call `announce` again.
- `up 72 2` runs only after `announce(7)` has returned. The `2` is `1 + 1`.
- `up 725 3` runs last among the `up` lines. `total 3` is that return value.
</details>


## Why it works this way

A call does not reuse the caller's local names. Python allocates a frame, stores `n` there, and points the frame at the line that is running. `digit_count(725)` stores 725. That frame then calls `digit_count(72)`, which stores 72 in a new frame. The frame that holds 725 is still on the stack, paused on `return 1 + ...`. The same split happens for 7. Only the base case returns a number. Each paused frame then adds one and returns. The trace table is that pause, written down.

| call | n in that frame | waiting for | returns |
| --- | --- | --- | --- |
| digit_count(725) | 725 | digit_count(72) | 3 |
| digit_count(72) | 72 | digit_count(7) | 2 |
| digit_count(7) | 7 | nothing; this is the base | 1 |

```
digit_count(725)     paused, will add 1
  digit_count(72)    paused, will add 1
    digit_count(7)   base, returns 1
```

Time follows the number of frames. A d-digit number makes d calls, and each call does a fixed amount of arithmetic, so the cost is proportional to d. Memory is the stack at its deepest moment, also proportional to d, because every frame stays alive until the base case returns. A loop that divides by 10 keeps a single frame and a single integer. The recursive form spends frames so the sentence "one digit plus the rest" is the shape of the code.

Python keeps the paused frames on purpose. A traceback has to name every call that is still waiting, and a call is allowed to do more work after the recursive call returns. The `print("up", ...)` line is that extra work. A language that threw the frame away before the recursive call could not run it, and could not show it in a traceback. The next lesson's limit is the bill for keeping those frames.

The cell below records `n` on the way in. The length of that list is the number of frames. It matches the digit count. The timed cell compares the same walk with `len(str(n))`. Timings move from run to run. The loop is faster. Recursion here is for the shape of the problem, not because it beats a flat loop on an integer.


In [3]:
def frames_for(n: int) -> list[int]:
    seen = []

    def visit(current: int) -> int:
        seen.append(current)
        if current < 10:
            return 1
        return 1 + visit(current // 10)

    count = visit(n)
    return [count, seen]


print(frames_for(725))
print(frames_for(8))

[3, [725, 72, 7]]
[1, [8]]


In [4]:
import timeit


def digit_count(n: int) -> int:
    if n < 10:
        return 1
    return 1 + digit_count(n // 10)


number = 10**18
recursive = timeit.timeit(lambda: digit_count(number), number=3000)
by_text = timeit.timeit(lambda: len(str(number)), number=3000)
print(round(recursive, 4), round(by_text, 4))

0.0026 0.0002


## When to use it

Use a recursive function when the problem is "one step, then the same problem on a smaller piece", and the depth stays small. Digit count is a clean example of the shape, and a weak example of the need: a loop is clearer and uses less memory. Nested data, in the next lesson, is where the shape earns its keep. Each part has the same shape as the whole, and a loop would have to manage a stack of positions itself.

Do not use recursion for a countdown, a total of a flat list, or anything whose depth can grow into the thousands. A `while` loop is the right tool there. If you already have a recursive function and the input might be a long chain, the lesson on the recursion limit shows how to move the same walk onto a list.

Do not point a recursive digit count at negatives or at a number you have not defined a base case for. The `if n < 10` test treats every negative as finished. Handle the sign first, or refuse it, before you divide.


## Common mistake

Forgetting to count the current digit still returns an answer. The answer is 1 for every input that eventually reaches a single digit. Nothing raises. The bug is that each frame throws away its own digit and returns whatever the deeper call returned.


In [5]:
def digit_count_wrong(n: int) -> int:
    if n < 10:
        return 1
    return digit_count_wrong(n // 10)


print(digit_count_wrong(725))
print(digit_count_wrong(8))
print(digit_count_wrong(10))

1
1
1


`725` prints `1`, and so does `10`. The frame for 725 returns the result of the frame for 72, which returns the result of the frame for 7, which is the base case. The `1 +` was the only place a frame added itself. The fix puts that addition back. The base case stays the same.


In [6]:
def digit_count(n: int) -> int:
    if n < 10:
        return 1
    return 1 + digit_count(n // 10)


print(digit_count(725))
print(digit_count(10))

3
2


A second mistake does raise. The recursive call receives the same `n` it was given, so the base case is never reached. The traceback repeats that call once per frame, and its last line is `RecursionError`. This cell catches the error and prints the name. Read a raw traceback from the last line upward: the name first, then the arrow on the line that called again without shrinking.


In [7]:
def shrink_never(n: int) -> int:
    if n < 10:
        return 1
    return 1 + shrink_never(n)


try:
    shrink_never(12)
except RecursionError as error:
    print(type(error).__name__)

RecursionError


## Try it

Change `digit_count` so it returns the digits themselves, most significant first: `725` becomes `[7, 2, 5]`. If you build the list after the recursive call, the deeper digits come back first. Which expression puts 7 at the front, and what list do you get if you append 7 after the recursive call instead of before it?


## Recap

- The base case returns without another self-call. The recursive case must shrink the input.
- Each call has its own frame. Later lines in a frame run only after the self-call returns.
- A missing `1 +` is silent and returns 1. A call that does not shrink raises `RecursionError`.

Next: [practice](01-base-case-practice.ipynb).
